# Task 2 — 80/10/10 single-fold model-selection check

Retrospective diagnostic on labeled Train: fit on 80%, select using the first 10%, inspect the second 10% once. All rows have appeared in previous research, so the final split is not independent confirmation. Test is not read.


In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score,mean_absolute_error
from IPython.display import display
SRC=ROOT/'src'


In [2]:
TASK=SRC/'task2';SAVE=TASK/'save';DATA='eda_original_v2'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path']);y=train.segment.astype(int)
from src.task2.models.decision_rules import restore_categorical_nulls
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
raw=train.drop(columns=['player_id','segment']);cats=raw.select_dtypes(include=['object']).columns.tolist()
X=restore_categorical_nulls(raw,cats)
fit_pos,temp_pos=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=SEED)
select_pos,check_pos=train_test_split(temp_pos,test_size=.5,stratify=y.iloc[temp_pos],random_state=SEED)
assert (len(fit_pos),len(select_pos),len(check_pos))==(83200,10400,10400)
results={}
def run(name,source):
 paths=[SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json']
 if any(p.exists() for p in paths):
  assert all(p.exists() for p in paths),f'incomplete {name}'
  card=json.loads(paths[1].read_text(encoding='utf-8'))
  assert card['data_name']==DATA and card['card']['source_model']==source
  model=joblib.load(paths[0])
 else:
  source_model=joblib.load(SAVE/f'{source}.joblib')
  model=Pipeline([('preprocess',clone(source_model.named_steps['preprocess'])),
                  ('classifier',CatBoostClassifier(**source_model.named_steps['classifier'].get_params()))])
  model.fit(X.iloc[fit_pos],y.iloc[fit_pos])
  save_model(task=2,name=name,model=model,data_name=DATA,
   card={'idea':'80/10/10 retrospective selection check','source_model':source,'seed':SEED,
         'fit_rows':len(fit_pos),'selection_rows':len(select_pos),'check_rows':len(check_pos)})
 selection_pred=np.asarray(model.predict(X.iloc[select_pos])).reshape(-1).astype(int)
 check_pred=np.asarray(model.predict(X.iloc[check_pos])).reshape(-1).astype(int)
 selection=fbeta_score(y.iloc[select_pos],selection_pred,beta=1,average='macro')
 check=fbeta_score(y.iloc[check_pos],check_pred,beta=1,average='macro')
 if not paths[2].exists():
  evaluate_model(task=2,name=name,score=float(check),metric_name='macro_F1_split801010_check',
   higher_is_better=True,card={'split':'retrospective single-fold 80/10/10 check',
   'selection_macro_F1':float(selection),'seed':SEED})
 else:
  saved=json.loads(paths[2].read_text(encoding='utf-8'));assert abs(saved['score']-check)<1e-10
 results[name]={'selection':selection,'check':check,'pred':check_pred}
 print(name,'selection',round(selection,6),'check',round(check,6))


## Fixed CatBoost candidates
No search is run on the check split. All preprocessing is refitted on the 80% split.


In [3]:
run('t2_split801010_baseline_v1','t2_cat_original_no_noise_v2')


t2_split801010_baseline_v1 selection 0.750925 check 0.749909


In [4]:
run('t2_split801010_weighted_v1','t2_cat_weighted_23_cv5_v4')


t2_split801010_weighted_v1 selection 0.748761 check 0.756953


In [5]:
run('t2_split801010_tuned_v1','t2_cat_tuned_cv5_v6')


t2_split801010_tuned_v1 selection 0.750969 check 0.751228


In [6]:
run('t2_split801010_tuned_weighted_v1','t2_cat_tuned_weighted_cv5_v7')


t2_split801010_tuned_weighted_v1 selection 0.749288 check 0.756516


## Selection versus check
Only the first 10% chooses a candidate.


In [7]:
table=pd.DataFrame([{'name':name,'selection':v['selection'],'check':v['check']} for name,v in results.items()])
table['selection_rank']=table.selection.rank(ascending=False,method='min').astype(int)
table['check_rank']=table.check.rank(ascending=False,method='min').astype(int)
display(table.sort_values('selection_rank').round(5))
chosen=table.sort_values('selection_rank').iloc[0]['name'];best=table.sort_values('check_rank').iloc[0]['name']
print('Selected on 10%:',chosen,'; best on check 10%:',best)
for a,b in [('t2_split801010_tuned_v1','t2_split801010_tuned_weighted_v1'),('t2_split801010_baseline_v1','t2_split801010_tuned_weighted_v1')]:
 pa,pb=results[a]['pred'],results[b]['pred'];print(a,'vs',b,'check disagreements',int((pa!=pb).sum()),'of',len(pa))
display(pd.DataFrame({name:pd.Series(v['pred']).value_counts().sort_index() for name,v in results.items()}))


,name,selection,check,selection_rank,check_rank
2,t2_split801010_tuned_v1,0.75097,0.75123,1,3
0,t2_split801010_baseline_v1,0.75093,0.74991,2,4
3,t2_split801010_tuned_weighted_v1,0.74929,0.75652,3,2
1,t2_split801010_weighted_v1,0.74876,0.75695,4,1


Selected on 10%: t2_split801010_tuned_v1 ; best on check 10%: t2_split801010_weighted_v1
t2_split801010_tuned_v1 vs t2_split801010_tuned_weighted_v1 check disagreements 298 of 10400
t2_split801010_baseline_v1 vs t2_split801010_tuned_weighted_v1 check disagreements 355 of 10400


,t2_split801010_baseline_v1,t2_split801010_weighted_v1,t2_split801010_tuned_v1,t2_split801010_tuned_weighted_v1
0,4536,4400,4481,4389
1,2580,2503,2595,2519
2,1859,2010,1886,2002
3,1425,1487,1438,1490


## Assessment

The selection split picks tuned unweighted CatBoost by 0.000044 macro F₁ over baseline. On the check split, weighted baseline ranks first (0.75695), tuned weighted second (0.75652), tuned unweighted third (0.75123), and baseline fourth (0.74991). The rank reversal is consistent with a class-balance tradeoff and single-split selection instability. It does not establish leakage or an independent new winner.
